In [36]:
import os
import cv2
import numpy as np
import random

In [40]:
DATASET_PATH = r"isolated_words_per_user"
IMG_SIZE = 64

char_data = {}

for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)

    if os.path.isdir(folder_path):
        char_data[folder] = []

        for file in os.listdir(folder_path):
            path = os.path.join(folder_path, file)
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                char_data[folder].append(img)

labels_list = sorted(char_data.keys())

print("Loaded classes:", len(labels_list))

Loaded classes: 82


In [41]:
OUTPUT_PATH = "yolo_dataset"
WORDS_TO_GENERATE = 800

os.makedirs(f"{OUTPUT_PATH}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/labels", exist_ok=True)

# base letters only (important)
base_letters = sorted(set([l.split('_')[0] for l in labels_list]))

for i in range(WORDS_TO_GENERATE):

    word_length = random.randint(2, 5)
    letters = random.choices(base_letters, k=word_length)

    canvas = np.zeros((IMG_SIZE, IMG_SIZE * word_length), dtype=np.uint8)

    boxes = []

    for idx, letter in enumerate(letters):

        possible = [k for k in char_data if k.startswith(letter)]
        char_label = random.choice(possible)

        img = random.choice(char_data[char_label])

        # realistic spacing
        shift = idx * random.randint(int(IMG_SIZE*0.5), int(IMG_SIZE*0.7))

        canvas[:, shift:shift+IMG_SIZE] = np.maximum(
            canvas[:, shift:shift+IMG_SIZE],
            img
        )

        # YOLO box
        x_center = (shift + IMG_SIZE/2) / canvas.shape[1]
        y_center = 0.5
        width = IMG_SIZE / canvas.shape[1]
        height = 1.0

        class_id = base_letters.index(letter)

        boxes.append(f"{class_id} {x_center} {y_center} {width} {height}")

    # add slight blur (important)
    canvas = cv2.GaussianBlur(canvas, (3,3), 0)

    cv2.imwrite(f"{OUTPUT_PATH}/images/word_{i}.png", canvas)

    with open(f"{OUTPUT_PATH}/labels/word_{i}.txt", "w") as f:
        f.write("\n".join(boxes))

print("Dataset generated successfully")

Dataset generated successfully


In [42]:
with open("dataset.yaml", "w") as f:
    f.write("path: yolo_dataset\n")
    f.write("train: images\n")
    f.write("val: images\n\n")
    f.write(f"nc: {len(base_letters)}\n\n")
    f.write("names:\n")

    for c in base_letters:
        f.write(f"  - {c}\n")

print("dataset.yaml ready")

dataset.yaml ready


In [43]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="dataset.yaml",
    epochs=10,
    imgsz=416
)

Ultralytics 8.4.23  Python-3.11.0 torch-2.10.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tr

KeyboardInterrupt: 

In [22]:
from ultralytics import YOLO

model = YOLO("runs/detect/train4/weights/last.pt")
model.train(
    data="dataset.yaml",
    epochs=10
)

Ultralytics 8.4.23  Python-3.11.0 torch-2.10.0+cpu CPU (Intel Core i7-10750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train4/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  4,  7,  8,  9, 11, 13, 14, 15, 17, 18, 19, 21, 22, 23, 25, 26, 27, 30, 31, 32, 34, 35, 36, 38, 39, 40, 42, 43, 44, 46, 48, 49, 50, 52, 53, 54, 56, 57, 59, 61, 62, 63])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000139CC36BAD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035

In [23]:
from ultralytics import YOLO
import cv2

model = YOLO("runs/detect/train4/weights/best.pt")

results = model("word_test.png")

original = cv2.imread("word_test.png")


image 1/1 f:\bach\Arabic-Handwritten-Characters-Recognition-using-CNN\word_test.png: 288x640 (no detections), 61.2ms
Speed: 2.9ms preprocess, 61.2ms inference, 1.6ms postprocess per image at shape (1, 3, 288, 640)


In [24]:
for box in results[0].boxes.xyxy:
    x1, y1, x2, y2 = map(int, box)

    char_img = original[y1:y2, x1:x2]

    char_img = cv2.resize(char_img, (32,32))
    char_img = char_img / 255.0
    char_img = char_img.reshape(1,32,32,1)

    pred = cnn_model.predict(char_img)

    idx = np.argmax(pred)
    label = CLASSES[idx]

    print(label)

In [29]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO

# ===== LOAD MODELS =====
yolo_model = YOLO("runs/detect/train4/weights/best.pt")  # adjust if train folder different
cnn_model = tf.keras.models.load_model("Deep_Learning.keras")

CLASSES = np.load("label_classes.npy")

# ===== IMAGE PATH =====
IMAGE_PATH = "word_test.png"  # change if needed

# ===== CHECK IMAGE EXISTS =====
print("Image exists:", os.path.exists(IMAGE_PATH))

# ===== RUN YOLO =====
results = yolo_model(IMAGE_PATH, save=True, conf=0.1)

# ===== LOAD ORIGINAL IMAGE =====
original = cv2.imread(IMAGE_PATH)

if original is None:
    print("❌ Error: Image not loaded")
else:
    print("✅ Image loaded successfully")

# ===== GET BOXES =====
boxes = results[0].boxes.xyxy.cpu().numpy()

print("Number of boxes detected:", len(boxes))

# ===== SORT RIGHT → LEFT (Arabic) =====
boxes = sorted(boxes, key=lambda x: x[0], reverse=True)

predicted_letters = []

for i, box in enumerate(boxes):
    x1, y1, x2, y2 = map(int, box)

    char_img = original[y1:y2, x1:x2]

    if char_img.size == 0:
        continue

    # ===== FIXED PREPROCESSING =====
    gray = cv2.cvtColor(char_img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    thresh = cv2.resize(thresh, (32, 32))

    processed = thresh / 255.0
    processed = processed.reshape(1, 32, 32, 1)

    # ===== DEBUG: SHOW INPUT TO CNN =====
    cv2.imshow(f"char_{i}", thresh)
    cv2.waitKey(300)

    # ===== PREDICT =====
    pred = cnn_model.predict(processed, verbose=0)

    idx = np.argmax(pred)
    confidence = np.max(pred)

    label = CLASSES[idx]
    base_letter = label.split('_')[0]

    print(f"{label} → {base_letter} ({confidence:.2%})")

    predicted_letters.append(base_letter)

# ===== FINAL WORD =====
print("\nFinal Word Prediction:")
print(" ".join(predicted_letters))

cv2.waitKey(0)
cv2.destroyAllWindows()

Image exists: True

image 1/1 f:\bach\Arabic-Handwritten-Characters-Recognition-using-CNN\word_test.png: 288x640 (no detections), 34.4ms
Speed: 1.3ms preprocess, 34.4ms inference, 0.4ms postprocess per image at shape (1, 3, 288, 640)
Results saved to F:\bach\Arabic-Handwritten-Characters-Recognition-using-CNN\runs\detect\predict
✅ Image loaded successfully
Number of boxes detected: 0

Final Word Prediction:

